# 자동 컨텍스트 컴팩션

오래 실행되는 에이전트 작업은 컨텍스트 한도를 넘기기 쉽습니다. 도구를 많이 쓰는 워크플로나 긴 대화는 토큰 컨텍스트 윈도를 금세 소진합니다. [AI 에이전트를 위한 효과적인 컨텍스트 엔지니어링](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)에서는 컨텍스트를 잘 관리하면 성능 저하와 컨텍스트 부패(context rot)를 피할 수 있다는 점을 다룬 바 있습니다.

Claude Agent Python SDK는 토큰 사용량이 설정한 임계치를 넘으면 대화 기록을 자동으로 압축해 이 컨텍스트를 관리해 줍니다. 덕분에 일반적인 200k 토큰 컨텍스트 한도를 넘어서도 작업을 이어 갈 수 있습니다.

이 쿡북에서는 **에이전트 기반 고객 상담 워크플로**를 통해 컨텍스트 컴팩션을 살펴봅니다. 지원 티켓 대기열을 처리하는 AI 고객 상담 에이전트를 만들었다고 상상해 보세요. 티켓마다 문제를 분류하고, 지식 베이스를 검색하고, 우선순위를 정하고, 알맞은 팀으로 배정하고, 답변 초안을 작성하고, 완료 처리해야 합니다. 티켓을 하나씩 처리해 나갈수록 대화 기록에는 분류 결과, 지식 베이스 검색 결과, 작성한 답변이 쌓이며 수천 토큰을 금세 소진합니다.

## 컨텍스트 컴팩션이란?

도구를 사용하는 에이전트 워크플로를 만들다 보면, 에이전트가 복잡한 작업을 반복하는 동안 대화가 매우 커질 수 있습니다. `compaction_control` 파라미터는 다음과 같은 방식으로 자동 컨텍스트 관리를 제공합니다.

1. 대화의 턴마다 토큰 사용량을 모니터링합니다
2. 임계치를 넘으면 요약 프롬프트를 사용자 턴으로 주입합니다
3. 모델이 `<summary></summary>` 태그로 감싼 요약을 생성하게 합니다. 이 태그는 파싱되지는 않지만 모델을 안내하는 역할을 합니다.
4. 대화 기록을 지우고 요약만 남긴 채 재개합니다
5. 압축된 컨텍스트로 작업을 이어 갑니다

## 이 쿡북을 마치면 다음을 할 수 있습니다.
 
 - 반복적인 워크플로에서 컨텍스트 한도를 효과적으로 관리하는 방법 이해하기
 - 자동 컨텍스트 컴팩션을 활용하는 에이전트 작성하기
 - 여러 번의 반복에도 초점을 유지하는 워크플로 설계하기

##  사전 준비

이 가이드를 따라 하기 전에 다음을 확인하세요.

**필요한 사전 지식**

- 에이전트 패턴과 도구 호출에 대한 기본 이해

**필요한 도구**

- Python 3.11 이상
- Anthropic API 키
- Anthropic SDK 0.74.1 이상

> **Opus 4.6을 쓰고 계신가요?** SDK 수준의 설정 없이 컨텍스트 관리를 자동으로 처리해 주는 [서버 측 컴팩션](https://docs.anthropic.com/en/docs/build-with-claude/compaction) 사용을 권장합니다.
>
> 이 쿡북은 **SDK 기반 컴팩션**을 다룹니다. 더 이전 모델을 사용하거나, 요약에 다른(더 저렴한) 모델을 쓰고 싶을 때 유용합니다.

## 준비

먼저 필요한 의존성을 설치합니다:

In [1]:
# %pip install -qU anthropic python-dotenv

참고: .env 파일에 다음 내용이 들어 있는지 확인하세요.

`ANTHROPIC_API_KEY=your_key_here`

환경 변수를 불러오고 클라이언트를 설정합니다. Claude 메시지 응답을 시각화하는 헬퍼 유틸리티도 함께 불러옵니다.

In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"

## 무대 설정하기

[utils/customer_service_tools.py](utils/customer_service_tools.py)에는 고객 지원 티켓을 처리하는 여러 함수를 정의해 두었습니다.

- `get_next_ticket()` — 대기열에서 아직 처리되지 않은 다음 티켓을 가져옵니다
- `classify_ticket(ticket_id, category)` — 문제를 결제, 기술, 계정, 제품, 배송 중 하나로 분류합니다
- `search_knowledge_base(query)` — 관련 도움말 문서와 해결책을 찾습니다
- `set_priority(ticket_id, priority)` — 우선순위(낮음, 보통, 높음, 긴급)를 지정합니다
- `route_to_team(ticket_id, team)` — 티켓을 알맞은 지원 팀으로 배정합니다
- `draft_response(ticket_id, response_text)` — 고객에게 보낼 답변을 작성합니다
- `mark_complete(ticket_id)` — 처리가 끝난 티켓을 완료 처리합니다

고객 상담 에이전트는 이 도구들로 티켓을 체계적으로 처리할 수 있습니다. 티켓마다 분류, 조사, 우선순위 지정, 배정, 답변 작성이 필요합니다. 티켓 20~30건을 연속으로 처리하면 모든 분류, 모든 지식 베이스 검색, 모든 작성 답변의 도구 결과가 대화 기록에 쌓이면서 토큰이 선형으로 증가합니다.

도구에 `beta_tool` 데코레이터를 붙이면 Claude 에이전트가 해당 도구를 사용할 수 있게 됩니다. 이 데코레이터는 함수 인자와 독스트링을 추출해 도구 메타데이터로 Claude에 제공합니다.

```python
import anthropic
from anthropic import beta_tool

@beta_tool
def get_next_ticket() -> dict:
    """Retrieve the next unprocessed support ticket from the queue."""
    ...
```

In [3]:
import anthropic
from utils.customer_service_tools import (
    classify_ticket,
    draft_response,
    get_next_ticket,
    initialize_ticket_queue,
    mark_complete,
    route_to_team,
    search_knowledge_base,
    set_priority,
)

client = anthropic.Anthropic()

tools = [
    get_next_ticket,
    classify_ticket,
    search_knowledge_base,
    set_priority,
    route_to_team,
    draft_response,
    mark_complete,
]

## 기준점: 컴팩션 없이 실행하기

현실적인 고객 상담 시나리오, 즉 지원 티켓 대기열 처리부터 시작해 보겠습니다.

워크플로는 다음과 같습니다.

**티켓마다:**
1. `get_next_ticket()`으로 티켓을 가져옵니다
2. 문제 범주를 분류합니다(결제, 기술, 계정, 제품, 배송)
3. 지식 베이스에서 관련 정보를 검색합니다
4. 알맞은 우선순위를 지정합니다(낮음, 보통, 높음, 긴급)
5. 올바른 팀으로 배정합니다
6. 고객 답변 초안을 작성합니다
7. 티켓을 완료 처리합니다
8. 다음 티켓으로 넘어갑니다

**난관**: 대기열에 티켓이 5건 있고 각각 7번의 도구 호출이 필요하므로, Claude는 35번 이상 도구를 호출하게 됩니다. 분류, 지식 베이스 검색, 작성한 답변 등 각 단계의 결과가 대화 기록에 누적됩니다. 컴팩션이 없으면 이 모든 데이터가 티켓마다 그대로 남아 있어서, 5번째 티켓에 이르면 컨텍스트에 앞선 티켓 4건의 상세 내용이 전부 들어 있게 됩니다.

먼저 이 워크플로를 **컴팩션 없이** 실행하고 어떤 일이 벌어지는지 살펴보겠습니다:

In [4]:
from anthropic.types.beta import BetaMessageParam

num_tickets = 5
initialize_ticket_queue(num_tickets)

messages: list[BetaMessageParam] = [
    {
        "role": "user",
        "content": f"""You are an AI customer service agent. Your task is to process support tickets from a queue.

For EACH ticket, you must complete ALL these steps:

1. **Fetch ticket**: Call get_next_ticket() to retrieve the next unprocessed ticket
2. **Classify**: Call classify_ticket() to categorize the issue (billing/technical/account/product/shipping)
3. **Research**: Call search_knowledge_base() to find relevant information for this ticket type
4. **Prioritize**: Call set_priority() to assign priority (low/medium/high/urgent) based on severity
5. **Route**: Call route_to_team() to assign to the appropriate team
6. **Draft**: Call draft_response() to create a helpful customer response using KB information
7. **Complete**: Call mark_complete() to finalize this ticket
8. **Continue**: Immediately fetch the next ticket and repeat

IMPORTANT RULES:
- Process tickets ONE AT A TIME in sequence
- Complete ALL 7 steps for each ticket before moving to the next
- Keep fetching and processing tickets until you get an error that the queue is empty
- There are {num_tickets} tickets total - process all of them
- Be thorough but efficient

Begin by fetching the first ticket.""",
    }
]

total_input = 0
total_output = 0
turn_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
)

for message in runner:
    messages_list = list(runner._params["messages"])
    turn_count += 1
    total_input += message.usage.input_tokens
    total_output += message.usage.output_tokens
    print(
        f"Turn {turn_count:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input:8,}"
    )

print(f"\n{'=' * 60}")
print("BASELINE RESULTS (NO COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count}")
print(f"Input tokens:  {total_input:,}")
print(f"Output tokens: {total_output:,}")
print(f"Total tokens:  {total_input + total_output:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,760 tokens | Output=  102 tokens | Messages= 3 | Cumulative In=   3,297
Turn  3: Input=  1,905 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,202
Turn  4: Input=  2,237 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,439
Turn  5: Input=  2,385 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,824
Turn  6: Input=  2,537 tokens | Output=  301 tokens | Messages=11 | Cumulative In=  12,361
Turn  7: Input=  2,888 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,249
Turn  8: Input=  3,079 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,328
Turn  9: Input=  3,316 tokens | Output=   91 tokens | Messages=17 | Cumulative In=  21,644
Turn 10: Input=  3,450 tokens | Output=   84 tokens | Messages=19 | Cumulative In=  25,094
Turn 11: Input=  3,777 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,871

기준점을 확보했으니, 컴팩션 없이 컨텍스트가 어떻게 늘어나는지 더 잘 파악할 수 있습니다. 보시다시피 턴마다 입력에 토큰이 더해지면서 선형으로 증가합니다.

그 결과 토큰 소비가 높아지고 컨텍스트 한도에도 금세 도달할 수 있습니다. 27번째 턴에 이르면 티켓 5건만 처리했는데도 누적 입력 토큰이 150,000에 달합니다.

컴팩션 없이 티켓 5건을 모두 처리한 뒤 Claude의 최종 응답을 확인해 보겠습니다:

In [5]:
print(message.content[-1].text)

---

## ✅ ALL TICKETS PROCESSED SUCCESSFULLY!

**Summary of Completed Work:**

I have successfully processed all 5 tickets from the queue. Here's what was accomplished:

1. **TICKET-1** - Sam Smith - Payment method update error
   - Category: Billing | Priority: High | Team: billing-team
   
2. **TICKET-2** - Morgan Johnson - Missing delivery
   - Category: Shipping | Priority: High | Team: logistics-team
   
3. **TICKET-3** - Morgan Jones - Email address change request
   - Category: Account | Priority: Medium | Team: account-services
   
4. **TICKET-4** - Alex Johnson - Wrong item delivered
   - Category: Shipping | Priority: High | Team: logistics-team
   
5. **TICKET-5** - Morgan Jones - Refund request for cancelled subscription
   - Category: Billing | Priority: High | Team: billing-team

Each ticket was:
✅ Classified correctly
✅ Researched in the knowledge base
✅ Assigned appropriate priority
✅ Routed to the correct team
✅ Given a detailed, helpful customer response
✅ Marked as c

### 문제 이해하기

위 기준점 워크플로에서 Claude는 다음을 수행해야 했습니다.
- **지원 티켓 5건**을 순차적으로 처리
- **티켓당 7단계** 수행(가져오기, 분류, 조사, 우선순위, 배정, 초안 작성, 완료)
- **도구 호출 35회**, 그 결과가 대화 기록에 누적
- **모든 분류, 모든 지식 베이스 검색, 모든 작성 답변**을 메모리에 보관

**왜 이런 일이 생길까요**:
1. **선형 토큰 증가** — 도구를 쓸 때마다 이전 도구 결과를 포함한 대화 기록 전체가 Claude에 전송됩니다
2. **컨텍스트 오염** — 티켓 B를 처리하는 동안에도 티켓 A의 분류와 답변 초안이 컨텍스트에 남아 있습니다
3. **비용 누적** — 5번째 티켓 차례가 되면 매 API 호출마다 앞선 티켓 4건의 데이터를 함께 보내게 됩니다
4. **느려지는 응답** — 거대한 컨텍스트를 처리하는 데 더 오랜 시간이 걸립니다
5. **한도 도달 위험** — 결국 200k 토큰 컨텍스트 윈도에 부딪힙니다

**실제로 필요한 것**: 티켓 A를 마친 뒤에 필요한 것은 **간단한 요약**(티켓 해결됨, 범주, 우선순위)뿐이며, 전체 분류 결과나 지식 베이스 검색 내용, 완성된 답변 초안 전문이 아닙니다. 상세한 작업 내역은 버리고 완료 요약만 남겨야 합니다.

자동 컨텍스트 컴팩션이 이 문제를 어떻게 해결하는지 살펴보겠습니다.

## 자동 컨텍스트 컴팩션 켜기

똑같은 고객 상담 워크플로를 이번에는 자동 컨텍스트 컴팩션을 켠 채로 실행해 보겠습니다. 도구 러너에 `compaction_control` 파라미터를 추가하기만 하면 됩니다.

`compaction_control` 파라미터에는 필수 필드 하나와 선택 필드 몇 개가 있습니다.

- **`enabled`**(필수): 컴팩션을 켜고 끄는 불리언 값
- **`context_token_threshold`**(선택): 컴팩션을 촉발하는 토큰 수(기본값: 100,000)
- **`model`**(선택): 요약에 사용할 모델(기본값은 메인 모델)
- **`summary_prompt`**(선택): 요약 생성을 위한 커스텀 프롬프트

이 고객 상담 워크플로에서는 **5,000 토큰 임계치**를 사용하겠습니다. 티켓 몇 건을 처리하고 나면 컴팩션이 자동으로 촉발된다는 뜻입니다. 이를 통해 Claude는 다음을 할 수 있습니다.
1. **완료 요약 유지**(해결한 티켓, 범주, 결과)
2. **상세한 도구 결과 폐기**(지식 베이스 문서 전문, 전체 분류 결과, 답변 초안 전문)
3. 다음 티켓 묶음을 처리할 때 **새로 시작**

이는 실제 상담원이 일하는 방식과 닮았습니다. 티켓을 해결하고, 간단히 기록하고, 다음 건으로 넘어가는 것이죠.

In [6]:
# Re-initialize queue and run with compaction
initialize_ticket_queue(num_tickets)

total_input_compact = 0
total_output_compact = 0
turn_count_compact = 0
compaction_count = 0
prev_msg_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
    compaction_control={
        "enabled": True,
        "context_token_threshold": 5000,
    },
)

for message in runner:
    turn_count_compact += 1
    total_input_compact += message.usage.input_tokens
    total_output_compact += message.usage.output_tokens
    messages_list = list(runner._params["messages"])
    curr_msg_count = len(messages_list)

    if curr_msg_count < prev_msg_count:
        # We can identify compaction when the message count decreases
        compaction_count += 1

        print(f"\n{'=' * 60}")
        print(f"🔄 Compaction occurred! Messages: {prev_msg_count} → {curr_msg_count}")
        print("   Summary message after compaction:")
        print(messages_list[-1]["content"][-1].text)  # type: ignore
        print(f"\n{'=' * 60}")

    prev_msg_count = curr_msg_count
    print(
        f"Turn {turn_count_compact:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input_compact:8,}"
    )

print(f"\n{'=' * 60}")
print("OPTIMIZED RESULTS (WITH COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count_compact}")
print(f"Compactions:   {compaction_count}")
print(f"Input tokens:  {total_input_compact:,}")
print(f"Output tokens: {total_output_compact:,}")
print(f"Total tokens:  {total_input_compact + total_output_compact:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,755 tokens | Output=  108 tokens | Messages= 3 | Cumulative In=   3,292
Turn  3: Input=  1,906 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,198
Turn  4: Input=  2,216 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,414
Turn  5: Input=  2,364 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,778
Turn  6: Input=  2,516 tokens | Output=  332 tokens | Messages=11 | Cumulative In=  12,294
Turn  7: Input=  2,898 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,192
Turn  8: Input=  3,090 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,282
Turn  9: Input=  3,325 tokens | Output=   97 tokens | Messages=17 | Cumulative In=  21,607
Turn 10: Input=  3,465 tokens | Output=   90 tokens | Messages=19 | Cumulative In=  25,072
Turn 11: Input=  3,801 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,873

자동 컨텍스트 컴팩션을 켜면 턴당 토큰 사용량이 선형으로 늘지 않고 컴팩션이 일어날 때마다 줄어드는 것을 볼 수 있습니다. 티켓을 처리하는 동안 컴팩션이 두 번 발생했고, 그 다음 턴에서 전체 토큰 사용량이 감소한 것이 확인됩니다.

기준점 버전과 비교하면 79,000 토큰만 사용했습니다. 컴팩션이 일어날 때마다 생성된 요약 메시지도 출력했는데, Claude가 앞선 티켓의 상세 내용을 어떻게 효과적으로 압축했는지 볼 수 있습니다.

컴팩션을 켠 상태로 티켓 5건을 모두 처리한 뒤의 최종 응답을 살펴보겠습니다.

In [7]:
print(message.content[-1].text)

Perfect! **ALL 5 TICKETS HAVE BEEN SUCCESSFULLY COMPLETED!** 🎉

## Final Summary - All Tickets Processed

### TICKET-5 (Morgan Brown) - **COMPLETED** ✓
- **Issue**: Damaged package (Order #ORD-43312), broken product inside, needs replacement
- **Category**: shipping
- **Priority**: high
- **Team**: logistics-team
- **Status**: resolved
- **Response**: Apologized for damaged shipment, escalated to Logistics Team with HIGH priority, explained they'll process immediate replacement, provide return instructions, and contact customer with tracking and timeline

---

## 🎯 ALL 5 TICKETS COMPLETED

1. ✅ **TICKET-1** (Chris Davis) - Account locked → account-services
2. ✅ **TICKET-2** (Chris Williams) - Billing charge → billing-team  
3. ✅ **TICKET-3** (John Jones) - Google Sheets integration → product-success
4. ✅ **TICKET-4** (Sam Johnson) - Plan comparison → product-success
5. ✅ **TICKET-5** (Morgan Brown) - Damaged shipment → logistics-team

### Processing Statistics
- **Total tickets process

### 결과 비교하기

컴팩션을 켜면 워크플로와 최종 요약의 품질은 유지하면서도 두 실행 사이에 토큰 절감의 뚜렷한 차이가 나타납니다.

자동 컨텍스트 컴팩션으로 달라진 점은 다음과 같습니다.

1. **티켓 몇 건마다 컨텍스트 초기화** — 티켓 5~7건을 처리해 도구 결과가 5k 토큰을 넘어서면 SDK가 자동으로 다음을 수행합니다.
   - 요약 프롬프트를 주입합니다
   - Claude가 `<summary></summary>` 태그로 감싼 완료 요약을 생성하게 합니다
   - 대화 기록을 지우고 상세 분류, 지식 베이스 검색, 답변을 폐기합니다
   - 완료 요약만 남긴 채 이어 갑니다

2. **입력 토큰이 일정 범위에 머무름** — 티켓을 처리할수록 100k 이상으로 누적되는 대신, 컴팩션 때마다 입력 토큰이 초기화됩니다. 5번째 티켓을 처리할 때 1~4번 티켓의 전체 도구 결과를 지고 가지 않습니다.

3. **작업이 성공적으로 완료됨** — 컨텍스트 한도에 부딪히지 않고 모든 티켓을 매끄럽게 처리합니다

4. **품질이 유지됨** — 요약이 핵심 정보를 보존합니다.
   - 처리한 티켓과 그 ID
   - 지정한 범주와 우선순위
   - 배정한 팀
   - 전체 진행 상태
   
   모든 티켓이 여전히 제대로 분류되고, 우선순위가 매겨지고, 배정되고, 답변됩니다.

5. **자연스러운 업무 흐름** — 실제 상담원이 일하는 방식과 같습니다. 티켓을 해결하고, 시스템에 간단히 기록하고, 닫고, 다음 건으로 넘어갑니다. 새 티켓을 처리하면서 모든 지식 베이스 문서와 답변 초안 전문을 계속 열어 두지는 않습니다.

토큰 절감을 시각화해 보겠습니다:

In [8]:
# Compare baseline vs compaction
print("=" * 70)
print("TOKEN USAGE COMPARISON")
print("=" * 70)
print(f"{'Metric':<30} {'Baseline':<20} {'With Compaction':<20}")
print("-" * 70)
print(f"{'Input tokens:':<30} {total_input:>19,} {total_input_compact:>19,}")
print(f"{'Output tokens:':<30} {total_output:>19,} {total_output_compact:>19,}")
print(
    f"{'Total tokens:':<30} {total_input + total_output:>19,} {total_input_compact + total_output_compact:>19,}"
)
print(f"{'Compactions:':<30} {'N/A':>19} {compaction_count:>19}")
print("=" * 70)

# Calculate savings
token_savings = (total_input + total_output) - (total_input_compact + total_output_compact)
savings_percent = (
    (token_savings / (total_input + total_output)) * 100 if (total_input + total_output) > 0 else 0
)

print(f"\n💰 Token Savings: {token_savings:,} tokens ({savings_percent:.1f}% reduction)")

TOKEN USAGE COMPARISON
Metric                         Baseline             With Compaction     
----------------------------------------------------------------------
Input tokens:                              204,416              82,171
Output tokens:                               4,422               4,275
Total tokens:                              208,838              86,446
Compactions:                                   N/A                   2

💰 Token Savings: 122,392 tokens (58.6% reduction)


## 컴팩션은 내부적으로 어떻게 동작하는가

`tool_runner`가 토큰 사용량이 임계치를 넘었다고 감지하면 자동으로 다음을 수행합니다.

1. 다음 API 호출 직전에 **워크플로를 일시 중단**합니다
2. 진행 상황을 요약해 달라는 **요약 요청을 사용자 메시지로 주입**합니다
3. **요약을 생성**합니다 — Claude가 `<summary></summary>` 태그로 감싼 요약을 만들며, 여기에는 다음이 담깁니다.
   - **완료한 티켓**: 해결한 티켓의 간단한 기록(ID, 범주, 우선순위, 결과)
   - **진행 상태**: 처리한 티켓 수와 남은 티켓 수
   - **주요 패턴**: 티켓들에서 눈에 띄는 경향
   - **다음 단계**: 이어서 할 일(남은 티켓 계속 처리)
4. **기록을 지웁니다** — 모든 도구 결과를 포함한 대화 기록 전체가 요약 하나로 대체됩니다
5. **처리를 재개합니다** — Claude가 압축된 컨텍스트로 다음 티켓 묶음을 이어서 처리합니다

## 컴팩션 설정 커스터마이징

사용 사례에 맞게 컴팩션 동작을 조정할 수 있습니다. 주요 설정 옵션은 다음과 같습니다.

### 임계치 조정하기

`context_token_threshold`는 컴팩션이 촉발되는 시점을 결정합니다:

```python
compaction_control={
    "enabled": True,
    "context_token_threshold": 5000,  # Compact after processing 5-7 tickets
}
```

임계치를 너무 낮게 잡으면 요약 자체가 다시 컴팩션을 촉발할 수 있으므로 주의해야 합니다. 여기서는 시연을 위해 5,000 토큰으로 설정했지만, 실제로는 여러 값을 실험해 워크플로에 가장 잘 맞는 값을 찾으세요.

일반적인 지침은 다음과 같습니다.

- **낮은 임계치(5k~20k)**: 
  - 경계가 뚜렷한 반복 작업 처리에 적합
  - 컴팩션이 더 자주 일어나고 컨텍스트 누적이 최소화됨
  - 대상을 순차적으로 처리하는 작업에 가장 적합
  
- **중간 임계치(50k~100k)**: 
  - 자연스러운 체크포인트가 적고 큰 다단계 워크플로
  - 컨텍스트 보존과 관리 사이의 균형
  - 비용이 큰 도구 호출이 있는 워크플로에 적합
  
- **높은 임계치(100k~150k)**: 
  - 상당한 과거 맥락이 필요한 작업
  - 컴팩션 빈도가 낮아 원시 상세 정보를 더 많이 보존
  - 호출당 비용은 높지만 컴팩션 횟수는 적음
  
- **기본값(100k)**: 일반적인 장시간 작업에 무난한 균형

**티켓 처리의 경우**: 티켓마다 상당한 양의 도구 결과가 생기지만 티켓끼리는 독립적이므로 5k 임계치가 잘 맞습니다. 티켓 A를 해결한 뒤 티켓 B를 처리할 때 A의 상세한 지식 베이스 검색 결과는 필요하지 않습니다.

### 요약에 다른 모델 사용하기

요약 생성에는 더 빠르고 저렴한 모델을 쓸 수도 있습니다:

```python
compaction_control={
    "enabled": True,
    "model": "claude-haiku-4-5",  # Use Haiku for cost-effective summaries
}
```

### 커스텀 요약 프롬프트

요약이 생성되는 방식을 안내하는 커스텀 프롬프트를 제공할 수 있습니다. 특정 유형의 정보를 반드시 보존해야 하는 고객 상담 워크플로에서 특히 유용합니다.

예를 들어 요구 사항에 맞춰 다음과 같은 프롬프트를 정의할 수 있습니다.
- 완료한 모든 티켓의 **티켓 요약**
- 지정한 **범주와 우선순위**
- **배정한 팀**
- **진행 상태**(완료한 티켓, 남은 티켓)
- 워크플로의 **다음 단계**

```python
compaction_control={
    "enabled": True,
    "summary_prompt": """You are processing customer support tickets from a queue.

Create a focused summary that preserves:

1. **COMPLETED TICKETS**: For each ticket you've fully processed:
   - Ticket ID and customer name
   - Issue category and priority assigned
   - Team routed to
   - Brief outcome

2. **PROGRESS STATUS**: 
   - How many tickets you've completed
   - Approximately how many remain in the queue

3. **NEXT STEPS**: Continue processing the next ticket

Format with clear sections and wrap in <summary></summary> tags."""
}
```

## 도구 없는 컴팩션: 단순 채팅 루프

위 예제들은 도구를 많이 쓰는 에이전트 워크플로에 초점을 맞췄지만, 컨텍스트 컴팩션은 사용자가 대화를 주도하는 **단순한 대화형 애플리케이션**에서도 유용합니다.

 **참고:** 위에서 살펴본 `compaction_control` 파라미터는 도구를 사용하는 에이전트 워크플로용 `tool_runner`와 함께 동작합니다. 도구가 없는 단순 채팅 애플리케이션에서는 같은 원리를 적용해 컴팩션을 직접 구현하게 됩니다.

사용자가 Claude와 긴 대화를 나누는 채팅 애플리케이션을 생각해 보세요. 복잡한 주제를 논의하거나, 아이디어를 다듬거나, 문제를 풀어 나가는 상황입니다. 대화가 길어지면 똑같은 컨텍스트 누적 문제에 부딪힙니다.

**차이점**: 토큰 증가를 일으키는 것이 도구 사용이 아니라 주고받는 대화 자체라는 점입니다. 교환할 때마다 메시지가 기록에 추가됩니다.
- 사용자가 질문합니다
- Claude가 상세한 답변을 제공합니다
- 사용자가 설명이나 부연을 요청합니다
- Claude가 더 많은 맥락과 함께 답변합니다
- 이것이 수십, 수백 번 반복됩니다

컴팩션이 없으면 50번째 턴에 이르러 매 API 호출마다 대화 기록 전체(50번의 교환 전부)를 보내게 됩니다.

**해결책**: 같은 패턴으로 채팅 루프에 컴팩션을 직접 구현합니다.
1. 턴마다 토큰 사용량을 추적합니다
2. 임계치를 넘으면 요약을 요청합니다
3. 대화 기록을 요약으로 대체합니다
4. 압축된 컨텍스트로 대화를 이어 갑니다

구현 방법을 살펴보겠습니다:

In [ ]:
#!/usr/bin/env python3
"""
Simple Compaction Example - User-Driven Chat Loop

This shows the basic pattern for a chat application with compaction.
No tools required - just a simple loop where the user drives continuation.
"""

# Configuration
COMPACTION_THRESHOLD = 3000  # Compact when tokens exceed this (low for demo purposes)

# Structured summarization prompt for compaction
SUMMARY_PROMPT = """You have been working on the task described above but have not yet completed it. Write a continuation summary that will allow you (or another instance of yourself) to resume work efficiently in a future context window where the conversation history will be replaced with this summary. Your summary should be structured, concise, and actionable. Include:

1. **Task Overview**
   - The user's core request and success criteria
   - Any clarifications or constraints they specified

2. **Current State**
   - What has been completed so far
   - Files created, modified, or analyzed (with paths if relevant)
   - Key outputs or artifacts produced

3. **Important Discoveries**
   - Technical constraints or requirements uncovered
   - Decisions made and their rationale
   - Errors encountered and how they were resolved
   - What approaches were tried that didn't work (and why)

4. **Next Steps**
   - Specific actions needed to complete the task
   - Any blockers or open questions to resolve
   - Priority order if multiple steps remain

5. **Context to Preserve**
   - User preferences or style requirements
   - Domain-specific details that aren't obvious
   - Any promises made to the user

Be concise but complete—err on the side of including information that would prevent duplicate work or repeated mistakes.
 Write in a way that enables immediate resumption of the task.

Wrap your summary in <summary></summary> tags."""

# Message history
messages = []

print("Chat with Claude (type 'quit' to exit, or just hit Enter to continue)")
print("This is a demonstration - try having a conversation and watch compaction trigger")
print("=" * 60)

# Simulate a conversation for demo purposes
demo_messages = [
    "Help me understand how Python decorators work",
    "Can you show me an example with a timing decorator?",
    "How would I make a decorator that takes arguments?",
]

for user_input in demo_messages:
    print(f"\nYou: {user_input}")

    # Add user message
    messages.append({"role": "user", "content": user_input})

    # Get Claude's response
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages,
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content,
        }
    )

    print("\nClaude: ", end="")
    for block in response.content:
        if block.type == "text":
            print(f"{block.text[:300]} ...")

    # Check if we should compact
    usage = response.usage

    # Calculate total tokens (includes cache tokens)
    total_input_tokens = (
        usage.input_tokens
        + (usage.cache_creation_input_tokens or 0)
        + (usage.cache_read_input_tokens or 0)
    )
    total_tokens = total_input_tokens + usage.output_tokens

    cache_info = ""
    if usage.cache_creation_input_tokens or usage.cache_read_input_tokens:
        cache_info = f" (cache: {usage.cache_creation_input_tokens or 0} write + {usage.cache_read_input_tokens or 0} read)"

    print(
        f"\n[Tokens: {total_input_tokens} in{cache_info} + {usage.output_tokens} out = {total_tokens} total]"
    )

    if total_tokens > COMPACTION_THRESHOLD:
        print(f"\n{'=' * 60}")
        print(f"🔄 Compacting conversation... {len(messages)} messages → ", end="", flush=True)

        # Get summary using structured prompt
        summary_response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            messages=messages + [{"role": "user", "content": SUMMARY_PROMPT}],
        )

        summary_text = "".join(
            block.text for block in summary_response.content if block.type == "text"
        )

        # Replace history with summary
        messages = [{"role": "user", "content": summary_text}]

        print("1 message")
        print(f"{'=' * 60}\n")

print(f"Final conversation messages: {messages[-1].get('content')}")

print("\nDemo complete! In a real application, this loop would continue with user input.")

Chat with Claude (type 'quit' to exit, or just hit Enter to continue)
This is a demonstration - try having a conversation and watch compaction trigger

You: Help me understand how Python decorators work


### 채팅 루프 패턴 이해하기

위 예제는 대화 맥락에서 컴팩션을 수동으로 구현한 것입니다. 동작 방식은 다음과 같습니다.

**핵심 구성 요소**:

1. **토큰 추적**: 응답마다 전체 토큰(입력 + 출력 + 캐시 토큰)을 계산합니다
2. **임계치 확인**: 합계가 임계치를 넘으면 컴팩션을 촉발합니다
3. **요약 요청**: 동일한 구조의 SUMMARY_PROMPT를 Claude에 보냅니다
4. **기록 대체**: 메시지 기록 전체를 요약 하나로 대체합니다
5. **계속 진행**: 다음 사용자 메시지는 전체 기록이 아니라 요약 위에서 이어집니다

**이 패턴을 쓸 때**:

- **긴 브레인스토밍 세션**: 사용자가 여러 턴에 걸쳐 Claude와 아이디어를 탐색할 때
- **학습 대화**: 수십 번의 교환에 걸친 튜토리얼이나 설명
- **반복적 개선**: 초안, 디자인, 해결책에 사용자가 피드백을 주는 경우
- **채팅 애플리케이션**: 멀티턴 대화 인터페이스 전반

**Tool Runner와의 주요 차이점**:

| 구분 | Tool Runner (자동) | 채팅 루프 (수동) |
|--------|------------------------|-------------------|
| **촉발** | 임계치 도달 시 자동 | 임계치 확인을 직접 구현 |
| **요약** | SDK가 요약 요청을 처리 | 직접 API를 호출 |
| **기록 관리** | SDK가 메시지를 대체 | 리스트를 직접 대체 |
| **사용 사례** | 도구를 쓰는 에이전트 워크플로 | 사용자가 주도하는 대화 |

**프로덕션 고려 사항**:

1. **임계치 조정**: 실제 애플리케이션에서는 더 큰 임계치를 사용하세요
2. **요약 프롬프트 커스터마이징**: 대화 유형(브레인스토밍, 기술 지원, 개인 교습 등)에 맞게 다듬으세요
3. **사용자에게 표시하기**: "대화를 요약하는 중..." 같은 메시지를 보여 주어 사용자가 잠시 멈추는 이유를 이해하게 하세요
4. **핵심 맥락 보존**: 사용자가 중요하게 여기는 도메인 정보를 요약 프롬프트가 담아내도록 하세요

이 패턴은 컴팩션이 언제 어떻게 일어날지를 완전히 제어할 수 있게 해 주므로, SDK의 자동 tool-runner 컴팩션을 쓸 수 없는 대화형 애플리케이션에 이상적입니다.

## 한계와 고려 사항

자동 컨텍스트 컴팩션은 강력하지만, 알아 두어야 할 중요한 한계가 있습니다.

### 서버 측 샘플링 루프

**현재의 한계**: 서버 측 웹 검색 도구처럼 서버 측 샘플링 루프를 사용하는 경우 컴팩션이 최적으로 동작하지 않습니다.

**이유**: 캐시 토큰이 샘플링 루프를 거치며 누적되어, 실제 대화 기록이 아니라 캐싱된 내용을 기준으로 컴팩션이 너무 일찍 촉발될 수 있습니다.

이 기능은 다음과 같은 경우에 가장 잘 동작합니다.
- ✅ 클라이언트 측 도구(이 쿡북의 고객 상담 API 같은 것)
- ✅ 일반적인 도구 사용이 있는 표준 에이전트 워크플로
- ✅ 파일 작업, 데이터베이스 질의, API 호출
- ❌ 서버 측 확장 사고(Extended Thinking)
- ❌ 서버 측 웹 검색 도구

### 정보 손실

**트레이드오프**: 요약은 본질적으로 일부 정보를 잃습니다. Claude가 핵심을 잘 짚어 내기는 하지만, 일부 세부 사항은 압축되거나 생략됩니다.

**티켓 처리에서는**: 
- ✅ **유지됨**: 티켓 ID, 범주, 우선순위, 팀, 결과, 진행 상태
- ❌ **손실됨**: 지식 베이스 문서 전문, 답변 초안 전문, 상세한 분류 근거

대개는 이 정도로 충분합니다. 모든 지식 베이스 문서와 답변 전문을 영원히 들고 있을 필요는 없고, 완료 기록만 있으면 됩니다.

**완화 방법**:
- 커스텀 요약 프롬프트로 핵심 정보를 보존하세요
- 광범위한 과거 맥락이 필요한 작업에는 임계치를 높게 설정하세요
- 각 단계가 원시 상세 정보가 아니라 요약 위에서 진행되도록 작업을 모듈화하세요

### 컴팩션을 쓰지 말아야 할 때

다음 경우에는 컴팩션을 피하세요.

1. **짧은 작업**: 작업이 50k~100k 토큰 안에 끝난다면 컴팩션은 불필요한 부담만 더합니다
2. **전체 감사 추적이 필요한 작업**: 이전의 모든 상세 정보에 접근해야 하는 작업도 있습니다
3. **서버 측 샘플링 워크플로**: 위에서 설명한 대로, 이 한계가 해소될 때까지 기다리세요
4. **고도로 반복적인 개선 작업**: 각 단계가 앞선 모든 단계의 정확한 세부 사항에 결정적으로 의존하는 경우

### 컴팩션을 써야 할 때

컴팩션은 다음 경우에 이상적입니다.

1. **순차 처리**: 이번 티켓 워크플로처럼 여러 항목을 차례로 처리하는 경우
2. **다단계 워크플로**: 각 단계가 진행 상황을 요약한 뒤 다음으로 넘어갈 수 있는 경우
3. **반복적 데이터 처리**: 대용량 데이터셋을 청크 단위로, 또는 대상을 하나씩 처리하는 경우
4. **긴 분석 세션**: 여러 대상에 걸쳐 데이터를 분석하는 경우
5. **배치 작업**: 서로 독립적인 항목 수백 개를 처리하는 경우

**티켓 처리는 완벽한 사례입니다.** 그 이유는 다음과 같습니다.
- 티켓별 워크플로가 대체로 독립적입니다
- 전체 도구 결과가 아니라 완료 요약만 있으면 됩니다
- 자연스러운 컴팩션 시점이 존재합니다(티켓 몇 건을 마친 뒤)
- 워크플로가 반복적이고 순차적입니다

## 요약

자동 컨텍스트 컴팩션은 오래 실행되는 에이전트 워크플로가 일반적인 컨텍스트 한도를 넘어설 수 있게 해 주는 강력한 기능입니다. 이 쿡북에서는 고객 상담 티켓 처리 워크플로를 통해 컴팩션을 살펴봤습니다.

### 다음 단계

여러분의 워크플로에도 컴팩션을 적용해 보세요.
1. 자연스러운 컴팩션 시점을 찾으세요(항목 처리 후, 단계 완료 후 등)
2. 항목별 경계가 뚜렷하다면 공격적인 임계치(5k~10k)로 시작하세요
3. 커스텀 요약 프롬프트로 핵심 정보를 보존하세요
4. 컴팩션이 언제 촉발되는지 모니터링하고 품질이 유지되는지 확인하세요
5. 필요에 맞게 임계치를 조정하세요

효과적인 컨텍스트 관리에 대해 더 알아보려면 [AI 에이전트를 위한 효과적인 컨텍스트 엔지니어링](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)을 참고하세요.